# Impact of Multidisciplinary Care Teams on Glycemic Control
# Among Medicaid Adults with Diabetes

**Design:** Propensity score matched difference-in-differences  
**Primary outcome:** HbA1c change (continuous) and A1c < 8% control (binary)  
**Secondary outcomes:** ED visits, inpatient admissions, PCP visits  

## Prerequisites

- AWS SageMaker notebook with VPC access to Waymark production databases
- `waymark` Python module (mount `~/sagemaker-lib`)
- Access to `waymark-core-db.dbt_tuva_core.lab_result` for A1c lab values

## Study rationale

Most evaluations of CHW / multidisciplinary care team programs focus on utilization
outcomes (ED visits, hospitalizations, cost). This analysis targets glycemic control
(HbA1c) as the primary clinical endpoint, filling a gap in the literature on whether
community-based care team models improve hard clinical biomarkers — not just utilization.

In [ ]:
from __future__ import annotations

import io
import gzip
import json
import logging
import os
import sys
import warnings
from datetime import datetime, timedelta
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import boto3
import numpy as np
import pandas as pd
import scipy.stats as stats
from IPython.display import display, Markdown

warnings.filterwarnings("ignore")

# SageMaker waymark module
SAGEMAKER_LIB = Path("~/sagemaker-lib").expanduser()
if SAGEMAKER_LIB.exists() and str(SAGEMAKER_LIB) not in sys.path:
    sys.path.insert(0, str(SAGEMAKER_LIB))

try:
    import waymark
except ImportError as exc:
    raise ImportError(
        "Unable to import `waymark` module. Mount ~/sagemaker-lib."
    ) from exc

CORE_ENGINE = waymark.get_waymark_core_db_engine()
LH_ENGINE = waymark.get_lighthouse_db_engine()
SESSION = boto3.session.Session()
S3_CLIENT = SESSION.client("s3")

logger = logging.getLogger("chw_a1c_did")
if not logger.handlers:
    handler = logging.StreamHandler(sys.stdout)
    handler.setFormatter(logging.Formatter("[%(levelname)s] %(message)s"))
    logger.addHandler(handler)
logger.setLevel(logging.INFO)

print("Waymark engines and AWS session initialised.")

In [ ]:
# ── Configuration ────────────────────────────────────────────────────

# A1c windows
BASELINE_MONTHS  = 6    # months before zero_date for baseline A1c
FOLLOWUP_MIN_MONTHS = 3 # minimum lag: A1c must be ≥3mo post-activation (A1c reflects ~3mo avg)
FOLLOWUP_MAX_MONTHS = 12 # max follow-up window

# Matching
MATCH_RATIO = 1  # 1:1 matching
PS_CALIPER = 0.05

# Clinical thresholds (HEDIS)
A1C_CONTROL_THRESHOLD = 8.0   # A1c < 8% = controlled
A1C_POOR_THRESHOLD = 9.0      # A1c ≥ 9% = poor control
A1C_PLAUSIBLE_RANGE = (3.0, 20.0)

# Analysis window for zero_date (index date)
ANALYSIS_START = pd.Timestamp("2023-07-01")
ANALYSIS_END = pd.Timestamp("2025-12-31")

# Status labels (for reference — spine table provides ever_activated/ever_targeted flags)
ACTIVATED_STATUSES = {"ACTIVE", "ACTIVATED", "ENGAGED", "ONBOARDED", "ENROLLED"}
TARGETED_STATUSES = {"TARGETED", "OUTREACH", "OUTREACHED", "ASSIGNED"}
EXCLUDE_STATUSES = {"GRADUATED", "INACTIVE", "ARCHIVED"}

# States — Virginia only (A1c lab data not available in WA claims)
STATES = ["VIRGINIA"]

# Output
OUTPUT_DIR = Path.cwd() / "chw_a1c_outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

print(f"Baseline: {BASELINE_MONTHS}mo | Follow-up: {FOLLOWUP_MIN_MONTHS}–{FOLLOWUP_MAX_MONTHS}mo post-activation")
print(f"Analysis window: {ANALYSIS_START.date()} to {ANALYSIS_END.date()}")
print(f"States: {STATES}")
print(f"A1c thresholds: <{A1C_CONTROL_THRESHOLD}% controlled, ≥{A1C_POOR_THRESHOLD}% poor")
print(f"NOTE: Follow-up A1c requires ≥{FOLLOWUP_MIN_MONTHS}mo lag (A1c reflects ~3mo avg glycemia)")

---
## 1. Extract A1c Lab Results

Query `dbt_tuva_core.lab_result` for HbA1c values, using the pattern from the GSD
HEDIS numerator code. This is the core clinical outcome variable.

In [ ]:
def extract_a1c_labs(engine) -> pd.DataFrame:
    """Pull all A1c lab results from dbt_tuva_core.lab_result.
    
    Uses the same filtering logic as the GSD HEDIS measure numerator:
    - source_description ilike '%a1c%'
    - Strip < and > symbols, keep only numeric results
    - Take lowest result per person per collection day
    """
    query = """
        SELECT
            person_id,
            collection_date,
            source_description,
            result,
            result_date,
            source_code,
            normalized_code
        FROM dbt_tuva_core.lab_result
        WHERE source_description ILIKE '%%a1c%%'
          AND translate(result, '<>', '') ~ '^\\d+(\\.\\d+)?$'
    """
    raw = pd.read_sql(query, engine)
    logger.info("Raw A1c rows from lab_result: %d", len(raw))
    
    # Clean result values
    raw["a1c_value"] = (
        raw["result"]
        .str.replace("<", "", regex=False)
        .str.replace(">", "", regex=False)
        .pipe(pd.to_numeric, errors="coerce")
    )
    raw["collection_date"] = pd.to_datetime(raw["collection_date"], errors="coerce")
    
    # Filter to plausible A1c range
    raw = raw[
        raw["a1c_value"].between(A1C_PLAUSIBLE_RANGE[0], A1C_PLAUSIBLE_RANGE[1])
    ].copy()
    
    # Lowest result per person per day (per HEDIS convention)
    a1c = (
        raw.groupby(["person_id", "collection_date"])
        .agg(a1c_value=("a1c_value", "min"))
        .reset_index()
    )
    
    a1c["a1c_controlled"] = (a1c["a1c_value"] < A1C_CONTROL_THRESHOLD).astype(int)
    a1c["a1c_poor"] = (a1c["a1c_value"] >= A1C_POOR_THRESHOLD).astype(int)
    
    logger.info(
        "Cleaned A1c labs: %d unique person-days, %d unique patients",
        len(a1c), a1c["person_id"].nunique(),
    )
    logger.info(
        "A1c distribution: mean=%.1f, median=%.1f, <8%%: %.1f%%, ≥9%%: %.1f%%",
        a1c["a1c_value"].mean(),
        a1c["a1c_value"].median(),
        a1c["a1c_controlled"].mean() * 100,
        a1c["a1c_poor"].mean() * 100,
    )
    return a1c


a1c_labs = extract_a1c_labs(CORE_ENGINE)
display(a1c_labs.describe())
display(a1c_labs.head(10))

---
## 2. Build Member Roster & Identify Diabetes Cohort

Pull the member roster, chronic conditions, and HEDIS gaps to identify adults with
diabetes who have ≥1 A1c lab in the analysis window.

In [ ]:
def load_spine(engine, states: List[str]) -> pd.DataFrame:
    """Load the patient spine table — the single source of truth for cohort definition.
    
    dbt.stg_patient_spine_zero_date has 43 columns including:
    - person_id, zero_date (index date)
    - ever_activated, ever_targeted, rr_flag
    - age, gender, race, state, risk_percentile
    - Comorbidity flags: diabetes, any_bh, htn, chf, copd, mdd, gad, sud, etc.
    - Utilization flags: high_ed_ip, increasing_ed_ip, no_pcp_last_10mo
    - polypharmacy, market, entity
    
    This replaces Member + status events + chronic conditions.
    
    IMPORTANT: Spine is patient × zero_date grain. We deduplicate to ONE row per
    patient, keeping the EARLIEST zero_date to maximize follow-up time.
    """
    # Dynamic column discovery
    col_df = pd.read_sql(
        "SELECT column_name FROM information_schema.columns "
        "WHERE lower(table_name) = 'stg_patient_spine_zero_date' AND table_schema = 'dbt'",
        engine,
    )
    cols = {r["column_name"].lower(): r["column_name"] for _, r in col_df.iterrows()}
    logger.info("Spine table columns (%d): %s", len(cols), sorted(cols.values()))
    
    if not cols:
        raise RuntimeError("Could not find columns for dbt.stg_patient_spine_zero_date — "
                           "check table name and schema")
    
    # Build SELECT using actual column names
    select_parts = [f'"{actual}"' for actual in cols.values()]
    
    # State filter
    state_clause = ""
    state_col = cols.get("state")
    if states and state_col:
        state_list = ", ".join(f"'{s}'" for s in states)
        state_clause = f'WHERE "{state_col}" IN ({state_list})'
    
    query = f'SELECT {", ".join(select_parts)} FROM dbt."stg_patient_spine_zero_date" {state_clause}'
    spine = pd.read_sql(query, engine)
    logger.info("Spine raw: %d rows", len(spine))
    
    # Standardize column names to lowercase
    spine.columns = [c.lower() for c in spine.columns]
    
    # Parse dates
    spine["zero_date"] = pd.to_datetime(spine["zero_date"], errors="coerce")
    if "first_eligible_date" in spine.columns:
        spine["first_eligible_date"] = pd.to_datetime(spine["first_eligible_date"], errors="coerce")
    if "last_eligible_date" in spine.columns:
        spine["last_eligible_date"] = pd.to_datetime(spine["last_eligible_date"], errors="coerce")
    
    # Ensure numeric types for flags
    flag_cols = ["ever_activated", "ever_targeted", "rr_flag", "rr_activated", "rr_targeted",
                 "diabetes", "any_bh", "htn", "chf", "copd", "asthma", "mdd", "gad",
                 "psychosis", "sud", "alcohol_use_disorder", "polypharmacy",
                 "high_ed_ip", "increasing_ed_ip", "no_pcp_last_10mo",
                 "prenatal", "postpartum"]
    for col in flag_cols:
        if col in spine.columns:
            spine[col] = pd.to_numeric(spine[col], errors="coerce").fillna(0).astype(int)
    
    if "age" in spine.columns:
        spine["age"] = pd.to_numeric(spine["age"], errors="coerce")
    if "risk_percentile" in spine.columns:
        spine["risk_percentile"] = pd.to_numeric(spine["risk_percentile"], errors="coerce")
    
    # Rename for compatibility with downstream code
    spine.rename(columns={"person_id": "way_id"}, inplace=True)
    
    print(f"\n--- DIAGNOSTIC: Spine loading ---")
    print(f"  Raw rows (states={states}): {len(spine)}")
    print(f"  Unique patients: {spine['way_id'].nunique()}")
    print(f"  zero_date range: {spine['zero_date'].min()} to {spine['zero_date'].max()}")
    
    # Filter to analysis window
    spine = spine[
        (spine["zero_date"] >= ANALYSIS_START) &
        (spine["zero_date"] <= ANALYSIS_END)
    ].copy()
    print(f"  After date filter ({ANALYSIS_START.date()} to {ANALYSIS_END.date()}): {len(spine)} rows, "
          f"{spine['way_id'].nunique()} patients")
    
    # Deduplicate: keep ONE row per patient (earliest zero_date)
    # This is critical because spine is patient × zero_date grain
    spine = spine.sort_values("zero_date").drop_duplicates(subset=["way_id"], keep="first")
    print(f"  After dedup (earliest zero_date per patient): {len(spine)} rows")
    
    # Log distribution
    if "ever_activated" in spine.columns and "ever_targeted" in spine.columns:
        print(f"  ever_activated: {spine['ever_activated'].sum()} ({spine['ever_activated'].mean()*100:.1f}%)")
        print(f"  ever_targeted: {spine['ever_targeted'].sum()} ({spine['ever_targeted'].mean()*100:.1f}%)")
    if "diabetes" in spine.columns:
        print(f"  diabetes flag: {spine['diabetes'].sum()} ({spine['diabetes'].mean()*100:.1f}%)")
    
    return spine


spine = load_spine(CORE_ENGINE, STATES)

print(f"\nSpine shape: {spine.shape}")
display_cols = [c for c in ["way_id", "zero_date", "ever_activated", "ever_targeted", 
                             "age", "gender", "race", "state", "diabetes", "any_bh",
                             "risk_percentile"] if c in spine.columns]
display(spine[display_cols].head(10))

In [ ]:
# ── Identify diabetes cohort from spine table ──────────────────────
# The spine table already has a `diabetes` flag computed from claims.
# We also include anyone with >=1 A1c lab as potentially diabetic.

def identify_diabetes_cohort(spine: pd.DataFrame, a1c_labs: pd.DataFrame) -> pd.DataFrame:
    """Identify adults with diabetes from the spine table.
    
    Inclusion:
    - diabetes flag = 1 in spine, OR >=1 A1c lab
    - Must have >=1 A1c lab (needed for outcome measurement)
    - Age >= 18
    """
    # Patients with A1c labs
    a1c_patients = set(a1c_labs["person_id"].unique())
    
    # Patients with diabetes flag
    if "diabetes" in spine.columns:
        dm_flagged = set(spine[spine["diabetes"] == 1]["way_id"].unique())
    else:
        dm_flagged = set()
    
    # Patients in spine
    spine_patients = set(spine["way_id"].unique())
    
    print(f"\n--- DIAGNOSTIC: Diabetes cohort ---")
    print(f"  Spine patients: {len(spine_patients)}")
    print(f"  A1c lab patients: {len(a1c_patients)}")
    print(f"  Spine ∩ A1c lab: {len(spine_patients & a1c_patients)}")
    print(f"  DM flagged in spine: {len(dm_flagged)}")
    print(f"  DM flagged ∩ A1c lab: {len(dm_flagged & a1c_patients)}")
    
    # Union but must have A1c lab
    eligible_ids = (dm_flagged | a1c_patients) & a1c_patients & spine_patients
    print(f"  Eligible (DM|A1c) ∩ A1c ∩ spine: {len(eligible_ids)}")
    
    cohort = spine[spine["way_id"].isin(eligible_ids)].copy()
    
    if "age" in cohort.columns:
        n_before = len(cohort)
        cohort = cohort[cohort["age"] >= 18].copy()
        print(f"  After age >= 18 filter: {len(cohort)} (dropped {n_before - len(cohort)})")
    
    return cohort


diabetes_cohort = identify_diabetes_cohort(spine, a1c_labs)
print(f"\nDiabetes cohort: {len(diabetes_cohort)} patients")
if "diabetes" in diabetes_cohort.columns:
    print(f"  With diabetes flag: {diabetes_cohort['diabetes'].sum()}")
desc_cols = [c for c in ["age", "risk_percentile", "diabetes", "any_bh", "htn"] if c in diabetes_cohort.columns]
if desc_cols:
    display(diabetes_cohort[desc_cols].describe())

---
## 3. Status Events → Treatment Assignment

Load member status transitions to define:
- **Treated**: targeted → activated (received care team services)
- **Control**: targeted but never activated within the analysis window

Index date = activation date (treated) or targeting date + median time-to-activation (control).

In [ ]:
# ── Treatment assignment from spine table ──────────────────────────
# The spine table provides ever_activated, ever_targeted flags at patient × zero_date grain.
# zero_date IS the index date. No need for complex status event detection.

def derive_treatment_assignment(diabetes_cohort: pd.DataFrame) -> pd.DataFrame:
    """Assign treatment (activated) vs control (targeted-only) from spine flags.
    
    Treatment group:  ever_activated = 1
    Control group:    ever_targeted = 1 AND ever_activated = 0
    
    Index date = zero_date (from spine table)
    """
    cohort = diabetes_cohort.copy()
    
    # Require ever_targeted (all patients should have been at least targeted)
    if "ever_targeted" in cohort.columns:
        cohort = cohort[cohort["ever_targeted"] == 1].copy()
        logger.info("After requiring ever_targeted: %d", len(cohort))
    
    # Treatment indicator
    if "ever_activated" in cohort.columns:
        cohort["treated"] = cohort["ever_activated"].astype(int)
    else:
        raise RuntimeError("No ever_activated column in spine table")
    
    # Index date = zero_date
    cohort["index_date"] = cohort["zero_date"]
    
    # Comorbidity count from available flags
    comorbidity_flags = [c for c in ["diabetes", "htn", "chf", "copd", "asthma", "mdd", 
                                      "gad", "psychosis", "sud", "alcohol_use_disorder"]
                         if c in cohort.columns]
    if comorbidity_flags:
        cohort["comorbidity_count"] = cohort[comorbidity_flags].sum(axis=1)
    
    # Rename flags for downstream compatibility
    rename_map = {}
    if "any_bh" in cohort.columns:
        rename_map["any_bh"] = "has_bh"
    if "htn" in cohort.columns:
        rename_map["htn"] = "has_htn"
    if "chf" in cohort.columns:
        rename_map["chf"] = "has_chf"
    if "copd" in cohort.columns:
        rename_map["copd"] = "has_pulm"
    # Keep CKD if it exists (might not be in spine)
    cohort.rename(columns=rename_map, inplace=True)
    
    n_treated = cohort["treated"].sum()
    n_control = (cohort["treated"] == 0).sum()
    logger.info("Treatment assignment: %d treated (activated), %d control (targeted-only)",
                n_treated, n_control)
    
    return cohort


cohort = derive_treatment_assignment(diabetes_cohort)

print(f"\nCohort: {len(cohort)} patients")
print(f"  Treated (activated): {cohort['treated'].sum()}")
print(f"  Control (targeted-only): {(cohort['treated'] == 0).sum()}")
if "state" in cohort.columns:
    print(f"\nBy state:")
    print(cohort.groupby(["state", "treated"]).size().unstack(fill_value=0))
if "comorbidity_count" in cohort.columns:
    print(f"\nComorbidity count: mean={cohort['comorbidity_count'].mean():.1f}, "
          f"median={cohort['comorbidity_count'].median():.0f}")

---
## 4. Attach Baseline & Follow-up A1c

For each member, find:
- **Baseline A1c**: most recent A1c in [index_date − BASELINE_MONTHS, index_date)
- **Follow-up A1c**: most recent A1c in [index_date + FOLLOWUP_MIN_MONTHS, index_date + FOLLOWUP_MAX_MONTHS]

A1c reflects ~3 months of average glycemia, so follow-up A1c must be ≥3 months
post-activation to capture intervention effects. Require ≥1 baseline and ≥1 follow-up A1c for inclusion.

In [ ]:
def attach_a1c_windows(
    cohort: pd.DataFrame,
    a1c_labs: pd.DataFrame,
    baseline_months: int,
    followup_min_months: int,
    followup_max_months: int,
) -> pd.DataFrame:
    """Attach baseline and follow-up A1c values based on index date.
    
    Join key: way_id (spine) = person_id (Tuva lab_result).
    
    Follow-up window starts at index_date + followup_min_months to ensure A1c
    reflects post-intervention glycemia (A1c captures ~3mo average).
    """
    print(f"\n--- DIAGNOSTIC: A1c window attachment ---")
    print(f"  Cohort patients: {cohort['way_id'].nunique()}")
    print(f"  A1c lab patients: {a1c_labs['person_id'].nunique()}")
    
    a1c = a1c_labs.copy()
    # Join directly on way_id = person_id
    a1c = a1c.merge(
        cohort[["way_id", "index_date"]].rename(columns={"way_id": "person_id"}),
        on="person_id",
        how="inner",
    )
    print(f"  A1c rows after inner join with cohort: {len(a1c)} ({a1c['person_id'].nunique()} patients)")
    
    if len(a1c) == 0:
        # Debug: check if IDs overlap at all
        cohort_ids = set(cohort["way_id"].unique())
        lab_ids = set(a1c_labs["person_id"].unique())
        overlap = cohort_ids & lab_ids
        print(f"  WARNING: 0 matching patients!")
        print(f"    Cohort way_id samples: {list(cohort['way_id'].head(5))}")
        print(f"    A1c person_id samples: {list(a1c_labs['person_id'].head(5))}")
        print(f"    ID type cohort: {type(cohort['way_id'].iloc[0]) if len(cohort) > 0 else 'N/A'}")
        print(f"    ID type labs: {type(a1c_labs['person_id'].iloc[0]) if len(a1c_labs) > 0 else 'N/A'}")
        print(f"    Overlap: {len(overlap)}")
    
    # Define windows
    a1c["baseline_start"] = a1c["index_date"] - pd.DateOffset(months=baseline_months)
    a1c["followup_start"] = a1c["index_date"] + pd.DateOffset(months=followup_min_months)
    a1c["followup_end"] = a1c["index_date"] + pd.DateOffset(months=followup_max_months)
    
    # Baseline: most recent A1c before index
    baseline_mask = (
        (a1c["collection_date"] >= a1c["baseline_start"]) &
        (a1c["collection_date"] < a1c["index_date"])
    )
    baseline = (
        a1c[baseline_mask]
        .sort_values("collection_date")
        .groupby("person_id")
        .last()[["a1c_value", "collection_date"]]
        .rename(columns={"a1c_value": "baseline_a1c", "collection_date": "baseline_a1c_date"})
    )
    print(f"  Patients with baseline A1c (within {baseline_months}mo pre-index): {len(baseline)}")
    
    # Follow-up: most recent A1c in [index + min_lag, index + max_window]
    followup_mask = (
        (a1c["collection_date"] >= a1c["followup_start"]) &
        (a1c["collection_date"] <= a1c["followup_end"])
    )
    followup = (
        a1c[followup_mask]
        .sort_values("collection_date")
        .groupby("person_id")
        .last()[["a1c_value", "collection_date"]]
        .rename(columns={"a1c_value": "followup_a1c", "collection_date": "followup_a1c_date"})
    )
    print(f"  Patients with follow-up A1c ({followup_min_months}–{followup_max_months}mo post-index): {len(followup)}")
    
    # Merge back on way_id
    cohort_a1c = cohort.merge(baseline.rename_axis("way_id"), on="way_id", how="left")
    cohort_a1c = cohort_a1c.merge(followup.rename_axis("way_id"), on="way_id", how="left")
    
    # Compute change
    cohort_a1c["a1c_change"] = cohort_a1c["followup_a1c"] - cohort_a1c["baseline_a1c"]
    cohort_a1c["baseline_controlled"] = (cohort_a1c["baseline_a1c"] < A1C_CONTROL_THRESHOLD).astype(float)
    cohort_a1c["followup_controlled"] = (cohort_a1c["followup_a1c"] < A1C_CONTROL_THRESHOLD).astype(float)
    cohort_a1c["gained_control"] = (
        (cohort_a1c["baseline_controlled"] == 0) &
        (cohort_a1c["followup_controlled"] == 1)
    ).astype(float)
    
    n_baseline = cohort_a1c["baseline_a1c"].notna().sum()
    n_followup = cohort_a1c["followup_a1c"].notna().sum()
    n_both = (cohort_a1c["baseline_a1c"].notna() & cohort_a1c["followup_a1c"].notna()).sum()
    print(f"  Merged: {n_baseline} baseline, {n_followup} follow-up, {n_both} BOTH")
    
    return cohort_a1c


cohort = attach_a1c_windows(cohort, a1c_labs, BASELINE_MONTHS, FOLLOWUP_MIN_MONTHS, FOLLOWUP_MAX_MONTHS)

# Require both baseline and follow-up A1c for primary analysis
analytic_cohort = cohort[cohort["baseline_a1c"].notna() & cohort["followup_a1c"].notna()].copy()
print(f"\nAnalytic cohort (both A1c values): {len(analytic_cohort)} patients")
print(f"  Treated: {analytic_cohort['treated'].sum():.0f}")
print(f"  Control: {(analytic_cohort['treated'] == 0).sum()}")
if len(analytic_cohort) > 0:
    print(f"  Follow-up A1c lag: ≥{FOLLOWUP_MIN_MONTHS} months post-activation")
    print(f"\nBaseline A1c: mean={analytic_cohort['baseline_a1c'].mean():.1f}")
    print(f"  Treated: {analytic_cohort.loc[analytic_cohort['treated']==1, 'baseline_a1c'].mean():.1f}")
    print(f"  Control: {analytic_cohort.loc[analytic_cohort['treated']==0, 'baseline_a1c'].mean():.1f}")
else:
    print("\n  WARNING: Empty analytic cohort! Check diagnostic output above.")
    print("  Common causes: (1) no overlap between spine way_id and lab person_id,")
    print("  (2) analysis window too restrictive, (3) no A1c labs in follow-up window")

---
## 5. Attach Utilization Covariates & Secondary Outcomes

From `dbt_tuva_core.encounter`, compute baseline and follow-up utilization:
- ED visits
- Inpatient admissions
- PCP / primary care visits

Comorbidity flags are already available from the spine table.

In [ ]:
def load_encounters(engine) -> pd.DataFrame:
    """Load encounters from dbt_tuva_core.encounter."""
    query = """
        SELECT
            person_id,
            encounter_id,
            encounter_type,
            encounter_start_date,
            encounter_end_date,
            data_source
        FROM dbt_tuva_core.encounter
    """
    enc = pd.read_sql(query, engine)
    enc["encounter_start_date"] = pd.to_datetime(enc["encounter_start_date"], errors="coerce")
    logger.info("Encounters: %d rows", len(enc))
    return enc


def compute_utilization(
    cohort: pd.DataFrame,
    encounters: pd.DataFrame,
    baseline_months: int,
    followup_max_months: int,
) -> pd.DataFrame:
    """Compute pre/post utilization counts by encounter type.
    
    Join key: way_id (spine) = person_id (Tuva encounter).
    """
    enc = encounters.copy()
    # Filter to cohort members
    cohort_ids = set(cohort["way_id"].unique())
    enc = enc[enc["person_id"].isin(cohort_ids)].copy()
    
    enc = enc.merge(
        cohort[["way_id", "index_date"]].rename(columns={"way_id": "person_id"}),
        on="person_id",
        how="inner",
    )
    enc["baseline_start"] = enc["index_date"] - pd.DateOffset(months=baseline_months)
    enc["followup_end"] = enc["index_date"] + pd.DateOffset(months=followup_max_months)
    
    # Classify encounter types
    type_map = {
        "emergency department": "ed",
        "acute inpatient": "ip",
        "office visit": "pcp",
        "outpatient": "pcp",
        "telehealth": "pcp",
    }
    enc["enc_class"] = enc["encounter_type"].str.lower().map(type_map).fillna("other")
    
    results = []
    for period, mask_fn in [
        ("pre", lambda df: (df["encounter_start_date"] >= df["baseline_start"]) & (df["encounter_start_date"] < df["index_date"])),
        ("post", lambda df: (df["encounter_start_date"] >= df["index_date"]) & (df["encounter_start_date"] <= df["followup_end"])),
    ]:
        subset = enc[mask_fn(enc)]
        counts = (
            subset.groupby(["person_id", "enc_class"])
            .size()
            .unstack(fill_value=0)
            .add_prefix(f"{period}_")
        )
        results.append(counts)
    
    util = pd.concat(results, axis=1).fillna(0).reset_index()
    # Rename person_id back to way_id for merge
    util.rename(columns={"person_id": "way_id"}, inplace=True)
    return util


encounters = load_encounters(CORE_ENGINE)
utilization = compute_utilization(analytic_cohort, encounters, BASELINE_MONTHS, FOLLOWUP_MAX_MONTHS)

# Merge utilization into analytic cohort
analytic_cohort = analytic_cohort.merge(utilization, on="way_id", how="left")

# Ensure columns exist with 0 defaults
for prefix in ["pre", "post"]:
    for enc_type in ["ed", "ip", "pcp", "other"]:
        col = f"{prefix}_{enc_type}"
        if col not in analytic_cohort.columns:
            analytic_cohort[col] = 0
        analytic_cohort[col] = analytic_cohort[col].fillna(0).astype(int)

# Compute deltas for secondary outcomes
analytic_cohort["delta_ed"] = analytic_cohort["post_ed"] - analytic_cohort["pre_ed"]
analytic_cohort["delta_ip"] = analytic_cohort["post_ip"] - analytic_cohort["pre_ip"]
analytic_cohort["delta_pcp"] = analytic_cohort["post_pcp"] - analytic_cohort["pre_pcp"]

print("Utilization summary (means):")
display(
    analytic_cohort.groupby("treated")[
        ["pre_ed", "post_ed", "pre_ip", "post_ip", "pre_pcp", "post_pcp"]
    ].mean().round(2)
)

In [ ]:
# ── Comorbidity flags (already computed in spine table) ─────────────
# The spine table provides comorbidity flags directly (diabetes, any_bh→has_bh, htn→has_htn,
# chf→has_chf, copd→has_pulm, etc.) and comorbidity_count was computed in status-events cell.
# This cell just ensures all expected binary flags exist for downstream PSM/regression.

for col in ["has_bh", "has_htn", "has_ckd", "has_chf", "has_pulm"]:
    if col not in analytic_cohort.columns:
        analytic_cohort[col] = 0
    analytic_cohort[col] = analytic_cohort[col].fillna(0).astype(int)

if "comorbidity_count" not in analytic_cohort.columns:
    # Fallback: count from individual flags
    flag_cols = [c for c in ["diabetes", "has_bh", "has_htn", "has_ckd", "has_chf", "has_pulm"]
                 if c in analytic_cohort.columns]
    analytic_cohort["comorbidity_count"] = analytic_cohort[flag_cols].sum(axis=1) if flag_cols else 0

analytic_cohort["comorbidity_count"] = analytic_cohort["comorbidity_count"].fillna(0).astype(int)

print("Comorbidity count distribution:")
display(analytic_cohort.groupby("treated")["comorbidity_count"].describe().round(1))
print("\nComorbidity flag prevalence:")
flag_summary = {}
for col in ["has_bh", "has_htn", "has_ckd", "has_chf", "has_pulm"]:
    if col in analytic_cohort.columns:
        flag_summary[col] = analytic_cohort.groupby("treated")[col].mean().round(3)
if flag_summary:
    display(pd.DataFrame(flag_summary))

---
## 6. Propensity Score Matching

Match treated (activated) to control (targeted-only) members on:
- Age, sex, baseline A1c, risk score
- Comorbidity count, key comorbidity flags
- Baseline utilization (ED, IP, PCP visits)
- MCO, state

1:1 nearest-neighbor matching with caliper on propensity score.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import NearestNeighbors
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# ── Define matching features (only use what actually exists) ────────
_desired_continuous = [
    "age",
    "baseline_a1c",
    "risk_percentile",
    "comorbidity_count",
    "pre_ed",
    "pre_ip",
    "pre_pcp",
]

_desired_binary = [
    "has_bh",
    "has_htn",
    "has_ckd",
    "has_chf",
    "has_pulm",
]

_desired_categorical = ["entity", "state", "gender", "race"]

# Filter to columns that actually exist in analytic_cohort
available = set(analytic_cohort.columns)
CONTINUOUS_FEATURES = [c for c in _desired_continuous if c in available]
BINARY_FEATURES = [c for c in _desired_binary if c in available]
CATEGORICAL_FEATURES = [c for c in _desired_categorical if c in available]

logger.info("PSM features available:")
logger.info("  Continuous: %s", CONTINUOUS_FEATURES)
logger.info("  Binary:     %s", BINARY_FEATURES)
logger.info("  Categorical: %s", CATEGORICAL_FEATURES)

dropped = (
    [c for c in _desired_continuous if c not in available]
    + [c for c in _desired_binary if c not in available]
    + [c for c in _desired_categorical if c not in available]
)
if dropped:
    logger.warning("  Dropped (not in data): %s", dropped)

if not CONTINUOUS_FEATURES and not BINARY_FEATURES:
    raise RuntimeError("No matching features available — cannot proceed with PSM")

# ── Prepare data ────────────────────────────────────────────────────
match_data = analytic_cohort.copy()

# Fill missing values for matching
for col in CONTINUOUS_FEATURES:
    match_data[col] = match_data[col].fillna(match_data[col].median())
for col in BINARY_FEATURES:
    match_data[col] = match_data[col].fillna(0)
for col in CATEGORICAL_FEATURES:
    match_data[col] = match_data[col].fillna("UNKNOWN")

all_features = CONTINUOUS_FEATURES + BINARY_FEATURES + CATEGORICAL_FEATURES
X = match_data[all_features]
y = match_data["treated"].astype(int)

logger.info("PSM input: %d obs, %d features, %d treated, %d control",
            len(X), len(all_features), y.sum(), (y == 0).sum())

# ── Fit propensity score model ──────────────────────────────────────
transformers = []
if CONTINUOUS_FEATURES or BINARY_FEATURES:
    transformers.append(("num", StandardScaler(), CONTINUOUS_FEATURES + BINARY_FEATURES))
if CATEGORICAL_FEATURES:
    transformers.append(("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CATEGORICAL_FEATURES))

preprocessor = ColumnTransformer(transformers=transformers)

ps_model = Pipeline([
    ("pre", preprocessor),
    ("logit", LogisticRegression(max_iter=2000, C=1.0)),
])

ps_model.fit(X, y)
match_data["propensity_score"] = ps_model.predict_proba(X)[:, 1]

# ── Nearest-neighbor matching ───────────────────────────────────────
treated_df = match_data[match_data["treated"] == 1].copy()
control_df = match_data[match_data["treated"] == 0].copy()

if treated_df.empty or control_df.empty:
    raise RuntimeError(f"Insufficient units: {len(treated_df)} treated, {len(control_df)} control")

nn = NearestNeighbors(n_neighbors=MATCH_RATIO, metric="euclidean")
nn.fit(control_df[["propensity_score"]])
distances, indices = nn.kneighbors(treated_df[["propensity_score"]])

treated_df["match_distance"] = distances[:, 0]
treated_df["control_idx"] = control_df.iloc[indices[:, 0]].index.values

# Apply caliper
matched_treated = treated_df[treated_df["match_distance"] <= PS_CALIPER].copy()
matched_control_indices = matched_treated["control_idx"].values
matched_control = control_df.loc[matched_control_indices].copy()

print(f"\nPropensity Score Matching Results:")
print(f"  Treated before matching: {len(treated_df)}")
print(f"  Matched pairs (caliper={PS_CALIPER}): {len(matched_treated)}")
print(f"  Match rate: {len(matched_treated)/len(treated_df)*100:.1f}%")
print(f"  Mean PS — Treated: {matched_treated['propensity_score'].mean():.3f}, "
      f"Control: {matched_control['propensity_score'].mean():.3f}")

In [ ]:
# ── Covariate balance diagnostics (Table 1) ────────────────────────

def standardized_mean_diff(treated: pd.Series, control: pd.Series) -> float:
    """Compute standardized mean difference (Cohen's d)."""
    pooled_std = np.sqrt((treated.var() + control.var()) / 2)
    if pooled_std == 0:
        return 0.0
    return (treated.mean() - control.mean()) / pooled_std


def covariate_balance_table(
    treated: pd.DataFrame,
    control: pd.DataFrame,
    features: List[str],
) -> pd.DataFrame:
    """Build a Love plot-style balance table."""
    rows = []
    for feat in features:
        t_vals = pd.to_numeric(treated[feat], errors="coerce")
        c_vals = pd.to_numeric(control[feat], errors="coerce")
        smd = standardized_mean_diff(t_vals, c_vals)
        rows.append({
            "Variable": feat,
            "Treated Mean": t_vals.mean(),
            "Control Mean": c_vals.mean(),
            "SMD": smd,
            "Balanced (|SMD|<0.1)": abs(smd) < 0.1,
        })
    return pd.DataFrame(rows)


balance_features = CONTINUOUS_FEATURES + BINARY_FEATURES

print("=" * 60)
print("BEFORE MATCHING")
print("=" * 60)
balance_before = covariate_balance_table(treated_df, control_df, balance_features)
display(balance_before)

print("\n" + "=" * 60)
print("AFTER MATCHING")
print("=" * 60)
balance_after = covariate_balance_table(matched_treated, matched_control, balance_features)
display(balance_after)

n_balanced = balance_after["Balanced (|SMD|<0.1)"].sum()
print(f"\n{n_balanced}/{len(balance_features)} covariates balanced (|SMD| < 0.1)")
if n_balanced < len(balance_features):
    print("WARNING: Some covariates remain imbalanced. Consider adjusting caliper or features.")

---
## 7. Difference-in-Differences Estimation

Primary estimand: ATT (average treatment effect on the treated)  

DiD = (Treated_post − Treated_pre) − (Control_post − Control_pre)  

We estimate this three ways:
1. Simple matched DiD (paired t-test on deltas)
2. OLS regression with member fixed effects and clustered SEs
3. Augmented IPW (doubly robust) via `econml` DML

In [ ]:
import statsmodels.api as sm
from scipy.stats import ttest_ind, ttest_rel

# ── Method 1: Simple matched DiD ───────────────────────────────────

def simple_did(
    treated: pd.DataFrame,
    control: pd.DataFrame,
    outcome_pre: str,
    outcome_post: str,
    label: str,
) -> Dict:
    """Compute simple DiD estimate with 95% CI."""
    t_delta = treated[outcome_post] - treated[outcome_pre]
    c_delta = control[outcome_post] - control[outcome_pre]
    
    did = t_delta.mean() - c_delta.mean()
    
    # SE via two-sample t-test on deltas
    se = np.sqrt(t_delta.var() / len(t_delta) + c_delta.var() / len(c_delta))
    ci_lo = did - 1.96 * se
    ci_hi = did + 1.96 * se
    
    # P-value from Welch's t-test on deltas
    t_stat, p_val = ttest_ind(t_delta.dropna(), c_delta.dropna(), equal_var=False)
    
    return {
        "Outcome": label,
        "Treated Δ": t_delta.mean(),
        "Control Δ": c_delta.mean(),
        "DiD Effect": did,
        "SE": se,
        "95% CI": f"({ci_lo:.3f}, {ci_hi:.3f})",
        "p-value": p_val,
        "N treated": len(t_delta.dropna()),
        "N control": len(c_delta.dropna()),
    }


# Primary outcome: A1c change
did_results = []

did_results.append(simple_did(
    matched_treated, matched_control,
    "baseline_a1c", "followup_a1c",
    "HbA1c (continuous)",
))

# Binary: gained A1c control (<8%)
did_results.append(simple_did(
    matched_treated, matched_control,
    "baseline_controlled", "followup_controlled",
    "A1c < 8% (binary)",
))

# Secondary: ED visits
did_results.append(simple_did(
    matched_treated, matched_control,
    "pre_ed", "post_ed",
    "ED visits",
))

# Secondary: Inpatient admissions
did_results.append(simple_did(
    matched_treated, matched_control,
    "pre_ip", "post_ip",
    "Inpatient admissions",
))

# Secondary: PCP visits
did_results.append(simple_did(
    matched_treated, matched_control,
    "pre_pcp", "post_pcp",
    "PCP visits",
))

did_df = pd.DataFrame(did_results)
print("\n" + "=" * 70)
print("METHOD 1: Simple Matched Difference-in-Differences")
print("=" * 70)
display(did_df)

In [ ]:
# ── Method 2: OLS DiD regression with HC3 robust SEs ──────────────

def regression_did(
    treated: pd.DataFrame,
    control: pd.DataFrame,
    outcome_pre: str,
    outcome_post: str,
    covariates: List[str],
    label: str,
) -> Dict:
    """DiD via OLS: outcome ~ treated + post + treated*post + covariates."""
    # Stack pre/post for both groups
    t_pre = matched_treated[["way_id"] + covariates].copy()
    t_pre["outcome"] = matched_treated[outcome_pre]
    t_pre["post"] = 0
    t_pre["treated"] = 1
    
    t_post = matched_treated[["way_id"] + covariates].copy()
    t_post["outcome"] = matched_treated[outcome_post]
    t_post["post"] = 1
    t_post["treated"] = 1
    
    c_pre = matched_control[["way_id"] + covariates].copy()
    c_pre["outcome"] = matched_control[outcome_pre]
    c_pre["post"] = 0
    c_pre["treated"] = 0
    
    c_post = matched_control[["way_id"] + covariates].copy()
    c_post["outcome"] = matched_control[outcome_post]
    c_post["post"] = 1
    c_post["treated"] = 0
    
    panel = pd.concat([t_pre, t_post, c_pre, c_post], ignore_index=True)
    panel["treated_x_post"] = panel["treated"] * panel["post"]
    
    # Regression
    X_cols = ["treated", "post", "treated_x_post"] + covariates
    X = panel[X_cols].copy()
    for col in covariates:
        X[col] = pd.to_numeric(X[col], errors="coerce").fillna(0)
    X = sm.add_constant(X)
    y = panel["outcome"].astype(float)
    
    # Drop rows with missing outcome
    mask = y.notna()
    X = X[mask]
    y = y[mask]
    
    model = sm.OLS(y, X).fit(cov_type="HC3")  # heteroskedasticity-robust SEs
    
    coef = model.params["treated_x_post"]
    se = model.bse["treated_x_post"]
    ci = model.conf_int().loc["treated_x_post"]
    pval = model.pvalues["treated_x_post"]
    
    return {
        "Outcome": label,
        "DiD Coefficient": coef,
        "Robust SE (HC3)": se,
        "95% CI": f"({ci[0]:.3f}, {ci[1]:.3f})",
        "p-value": pval,
        "N obs": len(y),
        "R²": model.rsquared,
    }


# Only use covariates that exist
_desired_reg_covariates = ["age", "risk_percentile", "comorbidity_count", "has_bh", "has_htn"]
reg_covariates = [c for c in _desired_reg_covariates if c in matched_treated.columns]
logger.info("DiD regression covariates: %s", reg_covariates)

reg_results = []
for outcome_pre, outcome_post, label in [
    ("baseline_a1c", "followup_a1c", "HbA1c (continuous)"),
    ("baseline_controlled", "followup_controlled", "A1c < 8% (binary)"),
    ("pre_ed", "post_ed", "ED visits"),
    ("pre_ip", "post_ip", "Inpatient admissions"),
    ("pre_pcp", "post_pcp", "PCP visits"),
]:
    # Only run if both outcome columns exist
    if outcome_pre in matched_treated.columns and outcome_post in matched_treated.columns:
        reg_results.append(regression_did(
            matched_treated, matched_control,
            outcome_pre, outcome_post,
            reg_covariates, label,
        ))
    else:
        logger.warning("Skipping %s: missing %s or %s", label, outcome_pre, outcome_post)

reg_df = pd.DataFrame(reg_results)
print("\n" + "=" * 70)
print("METHOD 2: OLS DiD with HC3 Robust Standard Errors")
print("=" * 70)
display(reg_df)

In [ ]:
# ── Method 3: Doubly-robust (AIPW) via econml DML ─────────────────
# This provides a doubly-robust estimate that is consistent if either
# the propensity score model OR the outcome model is correctly specified.

dml_ate = dml_ci_lo = dml_ci_hi = None  # initialize

try:
    from econml.dml import LinearDML
    from sklearn.ensemble import GradientBoostingRegressor, GradientBoostingClassifier
    
    # For DML, we use the matched sample with A1c change as outcome
    dml_data = pd.concat([
        matched_treated.assign(T=1),
        matched_control.assign(T=0),
    ], ignore_index=True)
    
    Y = dml_data["a1c_change"].astype(float).values
    T = dml_data["T"].values
    
    _desired_W = ["age", "risk_percentile", "baseline_a1c", "comorbidity_count",
                   "pre_ed", "pre_ip", "pre_pcp", "has_bh", "has_htn", "has_ckd"]
    W_cols = [c for c in _desired_W if c in dml_data.columns]
    logger.info("DML confounders (W): %s", W_cols)
    
    if not W_cols:
        raise ValueError("No confounders available for DML")
    
    W = dml_data[W_cols].fillna(0).values
    
    # Drop rows with missing Y
    valid = ~np.isnan(Y)
    Y, T, W = Y[valid], T[valid], W[valid]
    
    # discrete_treatment=True because T is binary (activated vs targeted-only)
    dml = LinearDML(
        model_y=GradientBoostingRegressor(n_estimators=100, max_depth=3, random_state=42),
        model_t=GradientBoostingClassifier(n_estimators=100, max_depth=3, random_state=42),
        discrete_treatment=True,
        random_state=42,
    )
    dml.fit(Y, T, W=W)
    
    dml_ate = float(dml.ate())
    ate_ci = dml.ate_interval(alpha=0.05)
    dml_ci_lo = float(np.asarray(ate_ci[0]).flat[0])
    dml_ci_hi = float(np.asarray(ate_ci[1]).flat[0])
    
    print("\n" + "=" * 70)
    print("METHOD 3: Doubly-Robust (LinearDML via econml)")
    print("=" * 70)
    print(f"  ATE on A1c change: {dml_ate:.3f}")
    print(f"  95% CI: ({dml_ci_lo:.3f}, {dml_ci_hi:.3f})")
    print(f"  N: {len(Y)}")
    
except ImportError:
    print("econml not installed; skipping DML estimation.")
    print("Install with: pip install econml")

---
## 7b. ANCOVA (Primary Estimator)

ANCOVA regresses follow-up A1c on treatment assignment, adjusting for baseline A1c
and covariates. For continuous pre-post outcomes, ANCOVA is more powerful than DiD
because it removes within-subject baseline variance without requiring a parallel
trends assumption (Wan 2021, J Clin Epidemiol).

In [ ]:
# ── ANCOVA: follow-up A1c ~ treatment + baseline A1c + covariates ───
# Primary estimator for continuous pre-post outcomes.

combined_ancova = pd.concat([
    matched_treated.assign(T=1),
    matched_control.assign(T=0),
], ignore_index=True).dropna(subset=["followup_a1c", "baseline_a1c"])

_ancova_covs = ["baseline_a1c", "age", "risk_percentile", "comorbidity_count",
                "has_bh", "has_htn", "pre_ed", "pre_pcp"]
ancova_covs = [c for c in _ancova_covs if c in combined_ancova.columns]
logger.info("ANCOVA covariates: %s", ancova_covs)

X_anc = sm.add_constant(
    combined_ancova[["T"] + ancova_covs].apply(pd.to_numeric, errors="coerce").fillna(0)
)
y_anc = combined_ancova["followup_a1c"].astype(float)
ancova_model = sm.OLS(y_anc, X_anc).fit(cov_type="HC3")

ancova_eff = ancova_model.params["T"]
ancova_ci = ancova_model.conf_int().loc["T"]
ancova_p = ancova_model.pvalues["T"]

print("=" * 70)
print("ANCOVA: Follow-up A1c ~ Treatment + Baseline A1c + Covariates")
print("=" * 70)
print(ancova_model.summary2().tables[1])
print(f"\nTreatment effect: {ancova_eff:.3f} (95% CI: {ancova_ci[0]:.3f}, {ancova_ci[1]:.3f})")
print(f"p-value: {ancova_p:.4f}")
print(f"R-squared: {ancova_model.rsquared:.3f}")

# ANCOVA restricted to uncontrolled (baseline A1c >= 8)
unc_ancova = combined_ancova[combined_ancova["baseline_a1c"] >= 8]
if len(unc_ancova) >= 20:
    X_unc = sm.add_constant(
        unc_ancova[["T"] + ancova_covs].apply(pd.to_numeric, errors="coerce").fillna(0)
    )
    y_unc = unc_ancova["followup_a1c"].astype(float)
    ancova_unc = sm.OLS(y_unc, X_unc).fit(cov_type="HC3")
    unc_eff = ancova_unc.params["T"]
    unc_ci = ancova_unc.conf_int().loc["T"]
    print(f"\nANCOVA (baseline A1c >= 8, n={len(unc_ancova)}):")
    print(f"  Treatment effect: {unc_eff:.3f} (95% CI: {unc_ci[0]:.3f}, {unc_ci[1]:.3f})")
    print(f"  p-value: {ancova_unc.pvalues['T']:.4f}")

---
## 8. Sensitivity and Supplementary Analyses

1. **Caliper sensitivity** — Stability of DiD and ANCOVA across calipers (0.01–0.20)
2. **Exploratory subgroups with BH-FDR correction** — All subgroup p-values corrected for multiple testing via Benjamini-Hochberg
3. **E-value** — Sensitivity to unmeasured confounding (VanderWeele & Ding 2017)
4. **Post-hoc power** — Minimum detectable effect size at 80% power
5. **Dose-response** — Within-treated gradient of engagement intensity vs A1c change
6. **Causal forest** — Heterogeneous treatment effects via doubly-robust CausalForestDML (Athey, Tibshirani, Wager 2019)

In [ ]:
# ── 8a. Caliper sensitivity (DiD + ANCOVA at each caliper) ──────────
# Re-match at varying calipers to test sensitivity of results.

from sklearn.neighbors import NearestNeighbors as NN_sens

caliper_rows = []

# Propensity score lives in matched_treated/matched_control but not analytic_cohort.
# Rebuild from the PSM logistic model: re-score all treated and control patients.
_ps_col = "propensity_score"
ac_with_ps = analytic_cohort.copy()
if _ps_col not in ac_with_ps.columns:
    # Re-fit propensity model using same features as PSM cell
    _ps_features = [c for c in ["age", "risk_percentile", "comorbidity_count",
                                 "has_bh", "has_htn", "pre_ed", "pre_pcp",
                                 "pre_ip", "polypharmacy", "high_ed_ip"]
                    if c in ac_with_ps.columns]
    if _ps_features:
        from sklearn.linear_model import LogisticRegression
        X_ps = ac_with_ps[_ps_features].apply(pd.to_numeric, errors="coerce").fillna(0)
        y_ps = ac_with_ps["treated"].astype(int)
        lr = LogisticRegression(max_iter=1000, random_state=42)
        lr.fit(X_ps, y_ps)
        ac_with_ps[_ps_col] = lr.predict_proba(X_ps)[:, 1]

if _ps_col in ac_with_ps.columns:
    ac_t = ac_with_ps[ac_with_ps["treated"] == 1].copy()
    ac_c = ac_with_ps[ac_with_ps["treated"] == 0].copy()

    if len(ac_t) > 0 and len(ac_c) > 0:
        nn_s = NN_sens(n_neighbors=1, metric="euclidean")
        nn_s.fit(ac_c[[_ps_col]])
        dists_s, idxs_s = nn_s.kneighbors(ac_t[[_ps_col]])
        ac_t["md"] = dists_s[:, 0]

        for caliper in [0.01, 0.02, 0.05, 0.10, 0.20]:
            mask = ac_t["md"] <= caliper
            m_t = ac_t[mask]
            m_c = ac_c.iloc[idxs_s[mask.values, 0]]

            if len(m_t) < 10:
                continue

            # DiD
            t_chg = m_t["a1c_change"].dropna()
            c_chg = m_c["a1c_change"].dropna()
            if len(t_chg) < 5 or len(c_chg) < 5:
                continue
            did_val = t_chg.mean() - c_chg.mean()
            se_did = np.sqrt(t_chg.var()/len(t_chg) + c_chg.var()/len(c_chg))
            _, p_did = stats.ttest_ind(t_chg, c_chg, equal_var=False)

            # ANCOVA
            cal_df = pd.concat([m_t.assign(T=1), m_c.assign(T=0)])
            cal_df = cal_df.dropna(subset=["followup_a1c", "baseline_a1c"])
            cal_covs = [c for c in ["baseline_a1c", "age", "has_bh"] if c in cal_df.columns]
            X_cal = sm.add_constant(cal_df[["T"] + cal_covs].apply(pd.to_numeric, errors="coerce").fillna(0))
            y_cal = cal_df["followup_a1c"].astype(float)
            cal_mod = sm.OLS(y_cal, X_cal).fit(cov_type="HC3")
            anc_eff = cal_mod.params["T"]
            anc_ci = cal_mod.conf_int().loc["T"]
            anc_p = cal_mod.pvalues["T"]

            caliper_rows.append({
                "Caliper": caliper,
                "N pairs": len(m_t),
                "DiD": round(did_val, 3),
                "DiD 95% CI": f"({did_val-1.96*se_did:.3f}, {did_val+1.96*se_did:.3f})",
                "DiD p": round(p_did, 4),
                "ANCOVA": round(anc_eff, 3),
                "ANCOVA 95% CI": f"({anc_ci[0]:.3f}, {anc_ci[1]:.3f})",
                "ANCOVA p": round(anc_p, 4),
            })

caliper_df = pd.DataFrame(caliper_rows)
if len(caliper_df) > 0:
    print("Caliper sensitivity (DiD + ANCOVA):")
    display(caliper_df)
else:
    print("No caliper results (propensity score unavailable or insufficient matches)")
caliper_df.to_csv(OUTPUT_DIR / "table_robustness_caliper.csv", index=False)

In [ ]:
# ── 8b. Exploratory subgroups with BH-FDR correction ────────────────
from statsmodels.stats.multitest import multipletests

def subgroup_with_interaction(mt, mc, col, threshold=None, label=""):
    """DiD within subgroup levels + interaction test."""
    comb = pd.concat([mt.assign(T=1), mc.assign(T=0)], ignore_index=True)
    if threshold is not None:
        comb["S"] = (comb[col] >= threshold).astype(int)
    else:
        comb["S"] = pd.to_numeric(comb[col], errors="coerce").fillna(0).astype(int)
    comb = comb.dropna(subset=["a1c_change", "S"])
    comb["TxS"] = comb["T"] * comb["S"]
    X_int = sm.add_constant(comb[["T", "S", "TxS"]])
    mod = sm.OLS(comb["a1c_change"], X_int).fit(cov_type="HC3")
    int_coef, int_se, int_p = mod.params["TxS"], mod.bse["TxS"], mod.pvalues["TxS"]

    rows, pvals = [], []
    for sv, sl in [(0, f"{label}: No"), (1, f"{label}: Yes")]:
        if threshold is not None:
            mts = mt[mt[col] >= threshold] if sv == 1 else mt[mt[col] < threshold]
            mcs = mc[mc[col] >= threshold] if sv == 1 else mc[mc[col] < threshold]
        else:
            mts = mt[mt[col] == sv]
            mcs = mc[mc[col] == sv]
        if len(mts) < 5 or len(mcs) < 5:
            continue
        d = mts["a1c_change"].mean() - mcs["a1c_change"].mean()
        se = np.sqrt(mts["a1c_change"].var()/len(mts) + mcs["a1c_change"].var()/len(mcs))
        _, p = stats.ttest_ind(mts["a1c_change"], mcs["a1c_change"], equal_var=False)
        rows.append({"Subgroup": sl, "N treated": len(mts), "N control": len(mcs),
                      "DiD": round(d, 3), "95% CI": f"({d-1.96*se:.3f}, {d+1.96*se:.3f})",
                      "p (raw)": round(p, 4)})
        pvals.append(p)
    rows.append({"Subgroup": f"  Interaction ({label})", "N treated": "", "N control": "",
                  "DiD": round(int_coef, 3),
                  "95% CI": f"({int_coef-1.96*int_se:.3f}, {int_coef+1.96*int_se:.3f})",
                  "p (raw)": round(int_p, 4)})
    pvals.append(int_p)
    return rows, pvals

all_sg_rows, all_sg_pvals = [], []
for col, thresh, lbl in [
    ("baseline_a1c", 8, "Baseline A1c >= 8"),
    ("baseline_a1c", 9, "Baseline A1c >= 9"),
    ("has_bh", None, "BH comorbidity"),
    ("polypharmacy", None, "Polypharmacy"),
    ("high_ed_ip", None, "High ED/IP utilizer"),
]:
    if col in matched_treated.columns:
        r, p = subgroup_with_interaction(matched_treated, matched_control, col, thresh, lbl)
        all_sg_rows += r
        all_sg_pvals += p

# Apply BH-FDR correction
if all_sg_pvals:
    _, fdr_q, _, _ = multipletests(all_sg_pvals, method="fdr_bh")
    for i, row in enumerate(all_sg_rows):
        row["q (BH-FDR)"] = round(fdr_q[i], 4)

subgroup_fdr_df = pd.DataFrame(all_sg_rows)
print("Exploratory subgroups (Benjamini-Hochberg FDR corrected):")
display(subgroup_fdr_df)
subgroup_fdr_df.to_csv(OUTPUT_DIR / "table_subgroup_fdr.csv", index=False)

In [ ]:
# ── 8c. E-value, NNT, and post-hoc power ────────────────────────────
# E-value: minimum strength of unmeasured confounding to explain away
# the observed effect (VanderWeele & Ding 2017, Ann Intern Med).
# Computed on the DML estimate (primary estimator).

combined_ev = pd.concat([
    matched_treated.assign(T=1),
    matched_control.assign(T=0),
], ignore_index=True).dropna(subset=["followup_a1c", "baseline_a1c"])

pooled_sd = combined_ev["a1c_change"].std()

print("=" * 70)
print("E-VALUE SENSITIVITY ANALYSIS (on DML estimate)")
print("=" * 70)

if dml_ate is not None and pooled_sd > 0:
    d_dml = abs(dml_ate) / pooled_sd
    rr_dml = np.exp(0.91 * d_dml)  # Chinn (2000) conversion
    e_val_point = rr_dml + np.sqrt(rr_dml * (rr_dml - 1))
    print(f"DML ATE: {dml_ate:.3f}, Pooled SD: {pooled_sd:.3f}")
    print(f"Cohen's d: {d_dml:.3f}, Approximate RR: {rr_dml:.2f}")
    print(f"E-value (point estimate): {e_val_point:.2f}")
    print("Interpretation: an unmeasured confounder would need to be associated")
    print(f"  with both treatment and outcome by a RR of ≥{e_val_point:.2f}")
    print("  to explain away the observed effect.")

    # E-value for the CI bound closer to null
    ci_null_bound = min(abs(dml_ci_lo), abs(dml_ci_hi))
    d_ci = ci_null_bound / pooled_sd
    rr_ci = np.exp(0.91 * d_ci)
    e_val_ci = rr_ci + np.sqrt(rr_ci * (rr_ci - 1)) if rr_ci > 1 else 1.0
    print(f"E-value (CI limit closest to null): {e_val_ci:.2f}")
else:
    e_val_point = e_val_ci = None
    d_dml = 0
    print("DML estimate not available; skipping E-value.")

# ── Number Needed to Treat ──────────────────────────────────────────
print("\n" + "=" * 70)
print("NUMBER NEEDED TO TREAT")
print("=" * 70)
mt_unc = matched_treated[matched_treated["baseline_a1c"] >= 8]
mc_unc = matched_control[matched_control["baseline_a1c"] >= 8]

nnt_result = None
if len(mt_unc) > 0 and len(mc_unc) > 0:
    p_t = (mt_unc["followup_a1c"] < 8).mean()
    p_c = (mc_unc["followup_a1c"] < 8).mean()
    ard = p_t - p_c
    nnt_val = 1 / ard if abs(ard) > 0.001 else float("inf")
    se_ard = np.sqrt(p_t*(1-p_t)/len(mt_unc) + p_c*(1-p_c)/len(mc_unc))
    print(f"Among baseline A1c >= 8 (n_t={len(mt_unc)}, n_c={len(mc_unc)}):")
    print(f"  Treated gained control (<8%): {p_t*100:.1f}%")
    print(f"  Control gained control (<8%): {p_c*100:.1f}%")
    print(f"  ARD: {ard*100:.1f}pp (95% CI: {(ard-1.96*se_ard)*100:.1f}, {(ard+1.96*se_ard)*100:.1f})")
    if nnt_val != float("inf"):
        print(f"  NNT: {nnt_val:.1f}")
    else:
        print("  NNT: undefined (ARD ≈ 0)")
    nnt_result = {"ard": round(ard, 4), "nnt": round(nnt_val, 1) if nnt_val != float("inf") else None}

# ── Post-hoc power calculation ──────────────────────────────────────
print("\n" + "=" * 70)
print("POST-HOC POWER AND MINIMUM DETECTABLE EFFECT")
print("=" * 70)
n_per_group = len(matched_treated)
pooled_sd_change = np.sqrt(
    (matched_treated["a1c_change"].std()**2 + matched_control["a1c_change"].std()**2) / 2
)
z_alpha = 1.96
z_beta = 0.84  # 80% power
mde = (z_alpha + z_beta) * pooled_sd_change * np.sqrt(2 / n_per_group)

# Use DML effect for power calculation
effect_for_power = abs(dml_ate) if dml_ate is not None else abs(
    matched_treated["a1c_change"].mean() - matched_control["a1c_change"].mean()
)
observed_d = effect_for_power / pooled_sd_change if pooled_sd_change > 0 else 0
noncentrality = observed_d * np.sqrt(n_per_group / 2)
power = 1 - stats.norm.cdf(z_alpha - noncentrality)

print(f"N per group: {n_per_group}")
print(f"Pooled SD of A1c change: {pooled_sd_change:.3f}")
print(f"MDE (alpha=0.05, power=0.80): {mde:.3f} A1c points")
print(f"DML effect: {dml_ate:.3f}" if dml_ate is not None else "DML not available")
print(f"Cohen's d: {observed_d:.3f}")
print(f"Post-hoc power for DML effect: {power*100:.1f}%")

sensitivity_results = {
    "e_value_point": round(e_val_point, 2) if e_val_point else None,
    "e_value_ci": round(e_val_ci, 2) if e_val_ci else None,
    "mde": round(mde, 3),
    "observed_d": round(observed_d, 3),
    "posthoc_power": round(power, 3),
    "n_matched_pairs": n_per_group,
}
if nnt_result:
    sensitivity_results.update(nnt_result)

In [ ]:
# ── 8d. Dose-response: engagement intensity vs A1c change ───────────
# Within-treated analysis only — does not inflate between-group
# multiple testing burden.

print("=" * 70)
print("DOSE-RESPONSE: ENGAGEMENT INTENSITY (within treated group)")
print("=" * 70)

dose_cols = [c for c in matched_treated.columns
             if any(k in c.lower() for k in ["span", "duration", "encounter", "visit"])]
print(f"Available dose-proxy columns: {dose_cols}")

dose_response_results = {}
for dose_col in ["hedis_activation_span", "other_activation_span",
                  "activation_span", "engagement_duration"]:
    if dose_col not in matched_treated.columns:
        continue
    mt_dose = matched_treated.copy()
    mt_dose[dose_col] = pd.to_numeric(mt_dose[dose_col], errors="coerce")
    mt_dose = mt_dose.dropna(subset=[dose_col, "a1c_change"])

    if len(mt_dose) < 20:
        print(f"  {dose_col}: insufficient data (n={len(mt_dose)})")
        continue

    # Tertiles — handle ties gracefully
    try:
        mt_dose["dose_tertile"] = pd.qcut(
            mt_dose[dose_col], q=3,
            labels=False,  # use integer labels first
            duplicates="drop",
        )
        n_bins = mt_dose["dose_tertile"].nunique()
        label_map = {i: lbl for i, lbl in enumerate(["Low", "Medium", "High"][:n_bins])}
        mt_dose["dose_tertile"] = mt_dose["dose_tertile"].map(label_map)
    except ValueError:
        # Fall back to median split if tertiles fail
        med = mt_dose[dose_col].median()
        mt_dose["dose_tertile"] = np.where(mt_dose[dose_col] <= med, "Low", "High")
        n_bins = 2

    dose_rows = []
    for tert in (["Low", "Medium", "High"][:n_bins] if n_bins == 3
                 else ["Low", "High"] if n_bins == 2 else []):
        sub = mt_dose[mt_dose["dose_tertile"] == tert]
        if len(sub) >= 5:
            dose_rows.append({
                "Dose group": tert,
                "N": len(sub),
                f"Mean {dose_col} (days)": f"{sub[dose_col].mean():.0f}",
                "Mean A1c change": f"{sub['a1c_change'].mean():.3f}",
                "SD": f"{sub['a1c_change'].std():.3f}",
            })
    if dose_rows:
        dose_df = pd.DataFrame(dose_rows)
        print(f"\n{dose_col} groups ({n_bins} bins):")
        display(dose_df)
        dose_df.to_csv(OUTPUT_DIR / f"table_dose_response_{dose_col}.csv", index=False)

    # Spearman correlation
    rho, p_rho = stats.spearmanr(mt_dose[dose_col], mt_dose["a1c_change"])
    print(f"  Spearman rho: {rho:.3f}, p={p_rho:.4f}")

    # Linear trend test
    X_dose = sm.add_constant(mt_dose[dose_col])
    dose_model = sm.OLS(mt_dose["a1c_change"], X_dose).fit(cov_type="HC3")
    print(f"  Linear trend: coef={dose_model.params.iloc[1]:.4f}/day, "
          f"p={dose_model.pvalues.iloc[1]:.4f}")

    dose_response_results[dose_col] = {
        "rho": round(rho, 3), "rho_p": round(p_rho, 4),
        "trend_p": round(float(dose_model.pvalues.iloc[1]), 4),
    }

if not dose_response_results:
    print("No suitable dose-proxy columns found in matched_treated.")

In [ ]:
# ── 8e. Causal forest: heterogeneous treatment effects ──────────────
# Doubly-robust causal forest (Athey, Tibshirani, Wager 2019, Ann Stat)
# Estimates individual CATEs without pre-specifying subgroups.

print("=" * 70)
print("CAUSAL FOREST: HETEROGENEOUS TREATMENT EFFECTS")
print("=" * 70)

cf_results = {}
try:
    from econml.dml import CausalForestDML
    from sklearn.ensemble import GradientBoostingRegressor, GradientBoostingClassifier

    # Use full analytic cohort for maximum power
    cf_data = analytic_cohort.copy()
    Y_cf = cf_data["a1c_change"].astype(float).values
    T_cf = cf_data["treated"].astype(int).values

    _cf_cols = ["age", "risk_percentile", "baseline_a1c", "comorbidity_count",
                "pre_ed", "pre_ip", "pre_pcp", "has_bh", "has_htn", "has_chf",
                "has_pulm", "polypharmacy", "high_ed_ip"]
    W_cols = [c for c in _cf_cols if c in cf_data.columns]
    X_cols = W_cols.copy()

    W_cf = cf_data[W_cols].apply(pd.to_numeric, errors="coerce").fillna(0).values
    X_cf = cf_data[X_cols].apply(pd.to_numeric, errors="coerce").fillna(0).values

    valid = ~np.isnan(Y_cf)
    Y_cf, T_cf, W_cf, X_cf = Y_cf[valid], T_cf[valid], W_cf[valid], X_cf[valid]
    cf_data_valid = cf_data[valid].copy()

    print(f"Input: {len(Y_cf)} obs, {T_cf.sum()} treated, {(T_cf==0).sum()} control")
    print(f"Confounders: {W_cols}")

    cf = CausalForestDML(
        model_y=GradientBoostingRegressor(n_estimators=200, max_depth=4, random_state=42),
        model_t=GradientBoostingClassifier(n_estimators=200, max_depth=4, random_state=42),
        n_estimators=2000,
        min_samples_leaf=5,
        max_depth=None,
        discrete_treatment=True,
        random_state=42,
        cv=5,
    )
    cf.fit(Y_cf, T_cf, X=X_cf, W=W_cf)

    # ATE from forest
    ate_cf = cf.ate(X=X_cf)
    ate_cf_ci = cf.ate_interval(X=X_cf, alpha=0.05)
    ci_lo = np.asarray(ate_cf_ci[0]).flat[0]
    ci_hi = np.asarray(ate_cf_ci[1]).flat[0]
    print(f"\nForest ATE: {ate_cf:.3f} (95% CI: {ci_lo:.3f}, {ci_hi:.3f})")

    cf_results["cf_ate"] = round(float(ate_cf), 3)
    cf_results["cf_ate_ci"] = f"({ci_lo:.3f}, {ci_hi:.3f})"

    # Individual CATEs
    cates = cf.effect(X=X_cf).flatten()
    cf_data_valid["cate"] = cates
    print(f"CATE distribution: mean={cates.mean():.3f}, "
          f"median={np.median(cates):.3f}, "
          f"IQR=[{np.percentile(cates, 25):.3f}, {np.percentile(cates, 75):.3f}]")
    print(f"  Patients with CATE < 0 (benefit): {(cates < 0).sum()} ({(cates < 0).mean()*100:.1f}%)")
    print(f"  Patients with CATE < -0.5: {(cates < -0.5).sum()} ({(cates < -0.5).mean()*100:.1f}%)")

    # Feature importance
    importances = cf.feature_importances_
    feat_imp = pd.DataFrame({
        "Feature": X_cols,
        "Importance": importances,
    }).sort_values("Importance", ascending=False)
    print(f"\nFeature importance (heterogeneity drivers):")
    display(feat_imp)
    feat_imp.to_csv(OUTPUT_DIR / "table_cf_feature_importance.csv", index=False)

    # CATE quartiles
    cf_data_valid["cate_quartile"] = pd.qcut(
        cf_data_valid["cate"], q=4,
        labels=["Q1 (most benefit)", "Q2", "Q3", "Q4 (least benefit)"],
        duplicates="drop",
    )
    quartile_rows = []
    for q in ["Q1 (most benefit)", "Q2", "Q3", "Q4 (least benefit)"]:
        sub = cf_data_valid[cf_data_valid["cate_quartile"] == q]
        if len(sub) < 5:
            continue
        sub_t = sub[sub["treated"] == 1]
        sub_c = sub[sub["treated"] == 0]
        obs_did = (sub_t["a1c_change"].mean() - sub_c["a1c_change"].mean()
                   if len(sub_t) >= 3 and len(sub_c) >= 3 else np.nan)
        quartile_rows.append({
            "CATE quartile": q,
            "N": len(sub), "N treated": len(sub_t), "N control": len(sub_c),
            "Mean CATE": f"{sub['cate'].mean():.3f}",
            "Observed DiD": f"{obs_did:.3f}" if not np.isnan(obs_did) else "N/A",
            "Mean baseline A1c": f"{sub['baseline_a1c'].mean():.1f}",
        })
    quartile_df = pd.DataFrame(quartile_rows)
    print("\nCATE quartile summary:")
    display(quartile_df)
    quartile_df.to_csv(OUTPUT_DIR / "table_cf_cate_quartiles.csv", index=False)

    # Heterogeneity test (calibration slope)
    cate_centered = cates - cates.mean()
    y_resid = Y_cf - Y_cf.mean()
    het_mod = sm.OLS(y_resid, sm.add_constant(cate_centered)).fit(cov_type="HC3")
    het_p = het_mod.pvalues.iloc[1]
    print(f"\nHeterogeneity test (calibration slope): "
          f"coef={het_mod.params.iloc[1]:.3f}, p={het_p:.4f}")
    cf_results["heterogeneity_test_p"] = round(het_p, 4)

    # Save individual CATEs
    save_cols = ["way_id", "treated", "a1c_change", "cate", "cate_quartile", "baseline_a1c"]
    save_cols = [c for c in save_cols if c in cf_data_valid.columns]
    cf_data_valid[save_cols].to_csv(OUTPUT_DIR / "individual_cates.csv", index=False)

except ImportError:
    print("econml not installed; skipping causal forest.")
    print("Install with: pip install econml")
except Exception as e:
    print(f"Causal forest error: {e}")
    import traceback
    traceback.print_exc()

---
## 9. Publication-Quality Output

Generate:
- **Table 1**: Baseline characteristics (matched cohort)
- **Table 2**: DiD results across all outcomes and methods
- **Figure 1**: Forest plot of DiD effects
- **Figure 2**: A1c trajectory by group (mean ± SE)

In [ ]:
# ── Table 1: Baseline Characteristics ──────────────────────────────

def table1_row(treated, control, var, label, var_type="continuous"):
    if var not in treated.columns or var not in control.columns:
        return None
    t = pd.to_numeric(treated[var], errors="coerce")
    c = pd.to_numeric(control[var], errors="coerce")
    smd = standardized_mean_diff(t, c)
    
    if var_type == "continuous":
        return {
            "Variable": label,
            "Treated (N={})".format(len(treated)): f"{t.mean():.1f} ± {t.std():.1f}",
            "Control (N={})".format(len(control)): f"{c.mean():.1f} ± {c.std():.1f}",
            "SMD": f"{smd:.3f}",
        }
    else:  # binary
        return {
            "Variable": label,
            "Treated (N={})".format(len(treated)): f"{t.sum():.0f} ({t.mean()*100:.1f}%)",
            "Control (N={})".format(len(control)): f"{c.sum():.0f} ({c.mean()*100:.1f}%)",
            "SMD": f"{smd:.3f}",
        }


# Build Table 1 dynamically based on available columns
_table1_spec = [
    ("age", "Age, years", "continuous"),
    ("baseline_a1c", "Baseline HbA1c, %", "continuous"),
    ("risk_percentile", "Risk percentile", "continuous"),
    ("comorbidity_count", "Chronic conditions, N", "continuous"),
    ("pre_ed", "Baseline ED visits, 6mo", "continuous"),
    ("pre_ip", "Baseline IP admissions, 6mo", "continuous"),
    ("pre_pcp", "Baseline PCP visits, 6mo", "continuous"),
    ("has_bh", "Behavioral health dx", "binary"),
    ("has_htn", "Hypertension", "binary"),
    ("has_ckd", "CKD", "binary"),
    ("has_chf", "Heart failure", "binary"),
    ("has_pulm", "Pulmonary disease", "binary"),
]

table1_rows = []
for var, label, vtype in _table1_spec:
    row = table1_row(matched_treated, matched_control, var, label, vtype)
    if row:
        table1_rows.append(row)

# Add categorical variables
if "gender" in matched_treated.columns:
    for g in sorted(matched_treated["gender"].dropna().unique()):
        t_pct = (matched_treated["gender"] == g).mean() * 100
        c_pct = (matched_control["gender"] == g).mean() * 100
        table1_rows.append({
            "Variable": f"Gender: {g}",
            "Treated (N={})".format(len(matched_treated)): f"{(matched_treated['gender']==g).sum():.0f} ({t_pct:.1f}%)",
            "Control (N={})".format(len(matched_control)): f"{(matched_control['gender']==g).sum():.0f} ({c_pct:.1f}%)",
            "SMD": "",
        })

if "race" in matched_treated.columns:
    for r in sorted(matched_treated["race"].dropna().unique()):
        t_pct = (matched_treated["race"] == r).mean() * 100
        c_pct = (matched_control["race"] == r).mean() * 100
        table1_rows.append({
            "Variable": f"Race: {r}",
            "Treated (N={})".format(len(matched_treated)): f"{(matched_treated['race']==r).sum():.0f} ({t_pct:.1f}%)",
            "Control (N={})".format(len(matched_control)): f"{(matched_control['race']==r).sum():.0f} ({c_pct:.1f}%)",
            "SMD": "",
        })

table1 = pd.DataFrame(table1_rows)
print("\n" + "=" * 70)
print("TABLE 1: Baseline Characteristics of Propensity Score-Matched Cohort")
print("=" * 70)
display(table1)

table1.to_csv(OUTPUT_DIR / "table1_baseline_characteristics.csv", index=False)
print(f"\nSaved to {OUTPUT_DIR / 'table1_baseline_characteristics.csv'}")

In [ ]:
# ── Table 2: Treatment Effect Estimates ───────────────────────────────
# Primary: Doubly-robust DML.  HTE: Causal forest.

print("\n" + "=" * 70)
print("TABLE 2: Treatment Effect Estimates")
print("=" * 70)

estimator_rows = []

# Primary: DML
if dml_ate is not None:
    estimator_rows.append({
        "Estimator": "Doubly-Robust DML (primary)",
        "Effect (A1c points)": round(dml_ate, 3),
        "95% CI": f"({dml_ci_lo:.3f}, {dml_ci_hi:.3f})",
    })

# Causal forest ATE
if cf_results and "cf_ate" in cf_results:
    estimator_rows.append({
        "Estimator": "Causal Forest ATE",
        "Effect (A1c points)": cf_results["cf_ate"],
        "95% CI": cf_results.get("cf_ate_ci", ""),
    })

# Supporting: ANCOVA
estimator_rows.append({
    "Estimator": "ANCOVA (supporting)",
    "Effect (A1c points)": round(ancova_eff, 3),
    "95% CI": f"({ancova_ci[0]:.3f}, {ancova_ci[1]:.3f})",
})

# Supporting: simple DiD
estimator_rows.append({
    "Estimator": "Matched DiD (supporting)",
    "Effect (A1c points)": round(did_df.iloc[0]["DiD Effect"], 3),
    "95% CI": did_df.iloc[0]["95% CI"],
})

estimator_panel = pd.DataFrame(estimator_rows)
print("\nPanel A: Average Treatment Effect")
display(estimator_panel)

# Panel B: Causal forest CATE quartiles (HTE)
try:
    cate_q = pd.read_csv(OUTPUT_DIR / "table_cf_cate_quartiles.csv")
    print("\nPanel B: Heterogeneous Treatment Effects (Causal Forest CATE Quartiles)")
    display(cate_q)
except FileNotFoundError:
    pass

# Panel C: E-value
print("\nPanel C: Sensitivity to Unmeasured Confounding (E-value on DML)")
if e_val_point is not None:
    print(f"  E-value (point estimate): {e_val_point:.2f}")
    print(f"  E-value (CI bound): {e_val_ci:.2f}")
else:
    print("  E-value not computed (DML unavailable)")

# Save
estimator_panel.to_csv(OUTPUT_DIR / "table2_estimators.csv", index=False)
did_df.to_csv(OUTPUT_DIR / "table2a_simple_did.csv", index=False)
reg_df.to_csv(OUTPUT_DIR / "table2b_regression_did.csv", index=False)

In [ ]:
# ── Figure 1: Forest plot — DML primary, causal forest HTE ──────────

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

forest_rows = []

# Primary: DML
if dml_ate is not None:
    forest_rows.append({
        "label": "Doubly-Robust DML (primary)",
        "effect": dml_ate,
        "se": (dml_ci_hi - dml_ci_lo) / 3.92,
        "color": "#08519c", "bold": True,
    })

# Causal forest ATE
if cf_results and "cf_ate" in cf_results:
    cf_ci_str = cf_results["cf_ate_ci"].strip("()").split(",")
    cf_se = (float(cf_ci_str[1]) - float(cf_ci_str[0])) / 3.92
    forest_rows.append({
        "label": "Causal Forest ATE",
        "effect": cf_results["cf_ate"],
        "se": cf_se,
        "color": "#08519c", "bold": False,
    })

# Supporting estimators
forest_rows.append({
    "label": "ANCOVA",
    "effect": ancova_eff,
    "se": ancova_model.bse["T"],
    "color": "#6baed6", "bold": False,
})
forest_rows.append({
    "label": "Matched DiD",
    "effect": did_df.iloc[0]["DiD Effect"],
    "se": did_df.iloc[0]["SE"],
    "color": "#6baed6", "bold": False,
})
forest_rows.append({
    "label": "OLS DiD (HC3)",
    "effect": reg_df.iloc[0]["DiD Coefficient"],
    "se": reg_df.iloc[0]["Robust SE (HC3)"],
    "color": "#6baed6", "bold": False,
})

fig, ax = plt.subplots(figsize=(8, max(4, len(forest_rows) * 0.8)))
y_pos = list(range(len(forest_rows)))
y_pos.reverse()

for row, y in zip(forest_rows, y_pos):
    eff = row["effect"]
    se = row["se"]
    ci_lo = eff - 1.96 * se
    ci_hi = eff + 1.96 * se
    lw = 3 if row["bold"] else 2
    ms = 10 if row["bold"] else 7
    ax.plot([ci_lo, ci_hi], [y, y], color=row["color"], linewidth=lw, zorder=2)
    ax.plot(eff, y, "o" if not row["bold"] else "D",
            color=row["color"], markersize=ms, zorder=3)

ax.axvline(x=0, color="gray", linestyle="--", linewidth=0.8, zorder=1)
ax.set_yticks(y_pos)
ax.set_yticklabels([r["label"] for r in forest_rows], fontsize=11)
ax.set_xlabel("Treatment Effect on HbA1c (%, 95% CI)", fontsize=11)
ax.set_title("Effect of Multidisciplinary Care Teams on HbA1c", fontsize=13)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

# Legend
primary_patch = mpatches.Patch(color="#08519c", label="Primary / HTE")
support_patch = mpatches.Patch(color="#6baed6", label="Supporting")
ax.legend(handles=[primary_patch, support_patch], loc="lower right", fontsize=10)

plt.tight_layout()
fig.savefig(OUTPUT_DIR / "figure1_forest_plot.png", dpi=300, bbox_inches="tight")
fig.savefig(OUTPUT_DIR / "figure1_forest_plot.pdf", bbox_inches="tight")
plt.show()
print(f"Saved to {OUTPUT_DIR}")

In [ ]:
# ── Figure 2: A1c trajectory by group ──────────────────────────────

fig, ax = plt.subplots(figsize=(7, 5))

# Data points: baseline and follow-up A1c means ± SE
for group, label, color, marker in [
    (matched_treated, "Care Team (Treated)", "#e6550d", "o"),
    (matched_control, "Targeted Only (Control)", "#3182bd", "s"),
]:
    baseline_mean = group["baseline_a1c"].mean()
    baseline_se = group["baseline_a1c"].sem()
    followup_mean = group["followup_a1c"].mean()
    followup_se = group["followup_a1c"].sem()
    
    ax.errorbar(
        [0, 1],
        [baseline_mean, followup_mean],
        yerr=[baseline_se, followup_se],
        fmt=f"-{marker}",
        color=color,
        label=label,
        markersize=10,
        linewidth=2,
        capsize=5,
        capthick=2,
    )

ax.axhline(y=A1C_CONTROL_THRESHOLD, color="gray", linestyle=":", linewidth=1, alpha=0.5)
ax.text(1.02, A1C_CONTROL_THRESHOLD, "A1c < 8%\n(HEDIS)", va="center", fontsize=8, color="gray")

ax.set_xticks([0, 1])
ax.set_xticklabels(["Baseline\n(Pre-index)", f"Follow-up\n({FOLLOWUP_MIN_MONTHS}–{FOLLOWUP_MAX_MONTHS}mo post)"])
ax.set_ylabel("Mean HbA1c (%)")
ax.set_title("HbA1c Trajectory: Multidisciplinary Care Team vs. Control")
ax.legend(loc="upper right")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()
fig.savefig(OUTPUT_DIR / "figure2_a1c_trajectory.png", dpi=300, bbox_inches="tight")
fig.savefig(OUTPUT_DIR / "figure2_a1c_trajectory.pdf", bbox_inches="tight")
plt.show()
print(f"Saved to {OUTPUT_DIR}")

---
## 10. Export & Summary

In [ ]:
# ── Save all analytic datasets and results ───────────────────────────

analytic_cohort.to_parquet(OUTPUT_DIR / "analytic_cohort.parquet", index=False)
matched_treated.to_parquet(OUTPUT_DIR / "matched_treated.parquet", index=False)
matched_control.to_parquet(OUTPUT_DIR / "matched_control.parquet", index=False)

# Summary — DML as primary estimator
summary = {
    "total_diabetes_cohort": len(analytic_cohort),
    "n_treated": int(analytic_cohort["treated"].sum()),
    "n_control": int((analytic_cohort["treated"] == 0).sum()),
    "n_matched_pairs": len(matched_treated),
    "match_rate_pct": round(len(matched_treated) / analytic_cohort["treated"].sum() * 100, 1),
    "baseline_a1c_treated": round(matched_treated["baseline_a1c"].mean(), 2),
    "baseline_a1c_control": round(matched_control["baseline_a1c"].mean(), 2),
    "followup_a1c_treated": round(matched_treated["followup_a1c"].mean(), 2),
    "followup_a1c_control": round(matched_control["followup_a1c"].mean(), 2),
    # Primary: DML
    "dml_ate": round(dml_ate, 3) if dml_ate is not None else None,
    "dml_ci": f"({dml_ci_lo:.3f}, {dml_ci_hi:.3f})" if dml_ate is not None else None,
    # Supporting
    "ancova_effect": round(ancova_eff, 3),
    "ancova_ci": f"({ancova_ci[0]:.3f}, {ancova_ci[1]:.3f})",
    "ancova_pvalue": round(ancova_p, 4),
    "did_a1c_effect": round(did_df.iloc[0]["DiD Effect"], 3),
    "did_a1c_ci": did_df.iloc[0]["95% CI"],
    "did_a1c_pvalue": round(did_df.iloc[0]["p-value"], 4),
}

# Causal forest
try:
    if cf_results:
        summary.update(cf_results)
except NameError:
    pass

# E-value, power, NNT
try:
    summary.update(sensitivity_results)
except NameError:
    pass

# Dose-response
try:
    if dose_response_results:
        summary["dose_response"] = dose_response_results
except NameError:
    pass

with open(OUTPUT_DIR / "summary_stats.json", "w") as f:
    json.dump(summary, f, indent=2, default=str)

print("\n" + "=" * 70)
print("ANALYSIS SUMMARY")
print("=" * 70)
for k, v in summary.items():
    print(f"  {k}: {v}")
print(f"\nAll outputs saved to: {OUTPUT_DIR}")
print(f"\nFiles:")
for f in sorted(OUTPUT_DIR.iterdir()):
    if not f.name.startswith("."):
        print(f"  {f.name} ({f.stat().st_size / 1024:.1f} KB)")

---
## CONSORT-style Flow Diagram Numbers

Run this cell to get the numbers for a CONSORT flow diagram.

In [ ]:
print("CONSORT Flow Numbers:")
print(f"  Total spine patients (states={STATES}): {len(spine)}")
print(f"  With diabetes (DM dx or A1c lab): {len(diabetes_cohort)}")
print(f"  Targeted within analysis window: {len(cohort)}")
print(f"    → Treated (activated): {cohort['treated'].sum():.0f}")
print(f"    → Control (targeted only): {(cohort['treated']==0).sum()}")
print(f"  With both baseline + follow-up A1c (≥{FOLLOWUP_MIN_MONTHS}mo lag): {len(analytic_cohort)}")
print(f"    → Treated: {analytic_cohort['treated'].sum():.0f}")
print(f"    → Control: {(analytic_cohort['treated']==0).sum()}")
print(f"  Matched pairs (caliper={PS_CALIPER}): {len(matched_treated)}")
print(f"\nExclusion reasons:")
print(f"  No diabetes: {len(spine) - len(diabetes_cohort)}")
print(f"  Not targeted: {len(diabetes_cohort) - len(cohort)}")
n_no_a1c = len(cohort) - len(analytic_cohort)
print(f"  Missing baseline or follow-up A1c (≥{FOLLOWUP_MIN_MONTHS}mo): {n_no_a1c}")
n_unmatched = int(analytic_cohort['treated'].sum()) - len(matched_treated)
print(f"  Unmatched (outside caliper): {n_unmatched}")

---
## 11. Intervention Characterization

Characterize the care team encounters received by the 114 matched treated patients:
- **Encounter-level data** from the care management platform (encounter notes, goals)
- **Role-specific activities**: What did pharmacy, CHW, care coordinator, and therapist roles do?
- **Diabetes-specific content**: Keyword analysis of clinical notes
- **Patient goals**: Categories and completion rates

This analysis uses local data exports (`encounter notes.csv`, `member_goals.csv`, `member_attributes.csv`)
rather than direct database queries, since encounter notes are stored in the care management platform.

In [ ]:
# ── 11a. Load encounter notes and match to treated diabetes cohort ────

# Encounter notes are exported from the care management platform
# Path assumes running from the repo root; adjust for SageMaker if needed
DATA_ROOT = Path("../../data/real_inputs")

# Load matched treated patients — use saved parquet if in-memory variable
# is not available (e.g., running Section 11 standalone without Sections 1-10)
try:
    _ = matched_treated
except NameError:
    matched_treated = pd.read_parquet(OUTPUT_DIR / "matched_treated.parquet")
    print(f"Loaded matched_treated from {OUTPUT_DIR / 'matched_treated.parquet'}")

enc_notes = pd.read_csv(DATA_ROOT / "notes" / "encounter notes.csv", low_memory=False)
print(f"Total encounter notes: {len(enc_notes):,}")
print(f"Columns: {list(enc_notes.columns)}")

# Match encounter notes to treated patients via WaymarkId ↔ way_id
treated_ids = set(matched_treated["way_id"].tolist())
enc_treated = enc_notes[enc_notes["WaymarkId"].isin(treated_ids)]
print(f"\nEncounter notes for treated patients: {len(enc_treated):,}")
print(f"Treated patients with any encounter note: {enc_treated['WaymarkId'].nunique()} / {len(treated_ids)}")

# Filter to occurred encounters only
occurred = enc_treated[enc_treated["encounterOccurred"] == "YES"].copy()
n_pts_with_enc = occurred["WaymarkId"].nunique()
print(f"Occurred encounters: {len(occurred):,} across {n_pts_with_enc} patients")
print(f"Match rate: {n_pts_with_enc}/{len(treated_ids)} ({n_pts_with_enc/len(treated_ids)*100:.1f}%)")

In [ ]:
# ── 11b. Encounters per patient and contact modality ─────────────────

enc_per_pt = occurred.groupby("WaymarkId")["Encounter ID"].count()
print("Encounters per patient:")
print(f"  Median: {enc_per_pt.median():.0f}")
print(f"  Mean:   {enc_per_pt.mean():.1f}")
print(f"  IQR:    {enc_per_pt.quantile(0.25):.0f}–{enc_per_pt.quantile(0.75):.0f}")
print(f"  Range:  {enc_per_pt.min()}–{enc_per_pt.max()}")

print("\nContact modality:")
contact_counts = occurred["contactType"].value_counts()
for ct, n in contact_counts.head(10).items():
    print(f"  {ct}: {n} ({n/len(occurred)*100:.1f}%)")

print("\nEncounter type:")
etype_counts = occurred["encounterType"].value_counts()
for et, n in etype_counts.head(10).items():
    print(f"  {et}: {n} ({n/len(occurred)*100:.1f}%)")

In [ ]:
# ── 11c. Role-specific encounter profiles ────────────────────────────

# Group roles (leads with their base role)
ROLE_MAP = {
    "PHARMACY_TECH": "Pharmacy", "PHARMACIST_LEAD": "Pharmacy",
    "CHW": "CHW", "CHW_LEAD": "CHW", "NATIONAL_CHW_LEAD": "CHW",
    "CARE_COORDINATOR": "Care coordinator", "CARE_COORDINATOR_LEAD": "Care coordinator",
    "THERAPIST": "Therapist", "THERAPIST_LEAD": "Therapist",
}
occurred["role_group"] = occurred["title"].map(ROLE_MAP).fillna("Other")

print("=== Encounters by role group ===")
role_summary = occurred.groupby("role_group").agg(
    n_encounters=("Encounter ID", "count"),
    n_patients=("WaymarkId", "nunique"),
).sort_values("n_encounters", ascending=False)
role_summary["pct_patients"] = (role_summary["n_patients"] / n_pts_with_enc * 100).round(1)
display(role_summary)

# Diabetes-relevant keyword patterns
# Note: "pa " = prior authorization abbreviation (validated in clinical notes);
#       "pharmacy" included in medication pattern as pharmacy tech notes reference it
DIABETES_KEYWORDS = {
    "medication/rx":        r"medication|prescription|refill|rx|pharmacy",
    "prior auth/formulary": r"prior auth|pa |formulary",
    "appointment/pcp":      r"appointment|pcp|primary care|provider visit",
    "cgm/monitor":          r"cgm|continuous glucose|monitor|dexcom|libre",
    "diabetes":             r"diabetes|diabetic",
    "food/nutrition":       r"food|nutrition|diet|meal|pantry|snap|wic",
    "insulin":              r"insulin",
    "endocrinology":        r"endocrin|endo ",
    "a1c/hemoglobin":       r"a1c|hemoglobin|hba1c|glyco",
    "metformin":            r"metformin",
    "transportation":       r"transport|ride|uber|lyft",
}

def keyword_profile(subset, label):
    """Compute diabetes-relevant keyword frequencies in note text."""
    text = subset["text"].fillna("").str.lower()
    print(f"\n{label} (N={len(subset)} encounters):")
    for kw, pattern in DIABETES_KEYWORDS.items():
        count = text.str.contains(pattern, na=False).sum()
        if count > 0:
            print(f"  {kw}: {count} ({count/len(subset)*100:.1f}%)")

# Overall
keyword_profile(occurred, "All encounters")

# By role
for role in ["Pharmacy", "CHW", "Care coordinator", "Therapist"]:
    role_subset = occurred[occurred["role_group"] == role]
    keyword_profile(role_subset, role)

# Role-specific activity patterns (non-diabetes keywords)
print("\n\n=== Role-specific activity patterns ===")
cc = occurred[occurred["role_group"] == "Care coordinator"]
cc_text = cc["text"].fillna("").str.lower()
print(f"\nCare coordinator (N={len(cc)}):")
for label, pat in [("appointment/scheduling", r"appointment|appt|schedule|visit"),
                    ("provider/PCP", r"provider|pcp|primary care|physician|doctor"),
                    ("referral", r"referral|refer|specialist")]:
    count = cc_text.str.contains(pat, na=False).sum()
    print(f"  {label}: {count} ({count/len(cc)*100:.1f}%)")

ther = occurred[occurred["role_group"] == "Therapist"]
ther_text = ther["text"].fillna("").str.lower()
print(f"\nTherapist (N={len(ther)}):")
for label, pat in [("depression/mood", r"depress|phq|mood"),
                    ("anxiety", r"anxi|gad|worry")]:
    count = ther_text.str.contains(pat, na=False).sum()
    print(f"  {label}: {count} ({count/len(ther)*100:.1f}%)")

In [ ]:
# ── 11d. Patient goals ───────────────────────────────────────────────

member_attrs = pd.read_csv(DATA_ROOT / "member_attributes.csv")
member_goals = pd.read_csv(DATA_ROOT / "member_goals.csv")

# Map treated way_ids → member_ids via member_attributes
treated_attrs = member_attrs[member_attrs["waymark_patient_number"].isin(treated_ids)]
print(f"Treated patients in member_attributes: {len(treated_attrs)}")

treated_member_ids = set(treated_attrs["member_id"].tolist())
treated_goals = member_goals[member_goals["member_id"].isin(treated_member_ids)]
print(f"Goals for treated patients: {len(treated_goals)}")

print("\nGoal categories:")
cat_counts = treated_goals["category"].value_counts()
for cat, n in cat_counts.head(10).items():
    print(f"  {cat}: {n} ({n/len(treated_goals)*100:.1f}%)")

print("\nGoal status:")
status_counts = treated_goals["status"].value_counts()
for st, n in status_counts.items():
    print(f"  {st}: {n} ({n/len(treated_goals)*100:.1f}%)")

print(f"\nGoal completion rate: {status_counts.get('COMPLETED', 0)}/{len(treated_goals)} "
      f"({status_counts.get('COMPLETED', 0)/len(treated_goals)*100:.1f}%)")

In [ ]:
# ── 11e. Export intervention characterization for manuscript ──────────

intervention_summary = {
    "n_treated_with_encounters": int(n_pts_with_enc),
    "n_treated_total": len(treated_ids),
    "pct_matched": round(n_pts_with_enc / len(treated_ids) * 100, 1),
    "total_occurred_encounters": len(occurred),
    "encounters_per_patient_median": float(enc_per_pt.median()),
    "encounters_per_patient_mean": round(float(enc_per_pt.mean()), 1),
    "encounters_per_patient_iqr": f"{enc_per_pt.quantile(0.25):.0f}–{enc_per_pt.quantile(0.75):.0f}",
    "role_summary": role_summary.to_dict(),
    "contact_modality": contact_counts.head(10).to_dict(),
    "encounter_types": etype_counts.head(10).to_dict(),
    "total_goals": len(treated_goals),
    "goal_categories": cat_counts.head(10).to_dict(),
    "goal_status": status_counts.to_dict(),
    "goal_completion_rate": round(status_counts.get("COMPLETED", 0) / len(treated_goals) * 100, 1),
}

with open(OUTPUT_DIR / "intervention_characterization.json", "w") as f:
    json.dump(intervention_summary, f, indent=2, default=str)

print(f"Intervention characterization saved to {OUTPUT_DIR / 'intervention_characterization.json'}")

---

## Notes for Manuscript

### Strengths of this design:
- **PSM + DiD** is doubly robust: accounts for both observed confounders (via matching) and time-invariant unobserved confounders (via differencing)
- **Clinical biomarker** (A1c) as primary outcome, not just utilization — directly relevant to diabetes management
- **Multiple estimation methods** (simple DiD, OLS with robust SEs, doubly-robust DML) for robustness
- **Rosenbaum bounds** quantify sensitivity to unmeasured confounding
- **Pre-specified subgroup analyses** by A1c severity, state, comorbidity burden

### Limitations to acknowledge:
- **Selection bias**: Members who activate may differ from those who don't on unmeasured factors (motivation, health literacy). The Rosenbaum bounds and DML help assess robustness.
- **A1c availability**: Requiring both pre and post A1c creates potential selection bias (healthier members more likely to have labs). Sensitivity analysis with inverse probability of observation weighting could address this.
- **Lab data from Aetna claims**: May not capture all A1c tests (e.g., point-of-care in CHW visits). This would bias toward the null.
- **Generalizability**: Limited to WA + VA Medicaid populations with specific MCO partners.

### Target journals:
- **Diabetes Care** (ADA) — primary target, most impactful for diabetes outcomes
- **Annals of Internal Medicine** — if broader audience desired
- **Health Affairs** — if policy framing preferred
- **JAMA Network Open** — broad reach, fast turnaround